# 04 — Fine Wine Heterogeneity & Wine-Level Price Behaviour During Market Stress

**Purpose**: Demonstrate that fine wine is highly heterogeneous — individual wines behave very
differently from each other and from market indices during downturns.

**Target wines** (LWIN7): Salon (1807626), Dom Perignon (1082656), Lafite (1011872)

## Sections
1. Environment setup
2. MotherDuck connection & schema discovery
3. Build per-wine monthly price series (SQL-side aggregation, 750ml, VWAP)
4. Load comparison assets (S&P 500, Liv-ex 100)
5. Per-wine price series chart
6. Drawdown analysis: 2008 GFC
7. Trade volume alongside prices
8. Best & worst performers during market stress periods
9. Data quality assertions

## 1. Environment Setup

In [ ]:
import os
import warnings
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
REPO_ROOT  = Path("__file__").resolve().parent.parent
DATA_DIR   = REPO_ROOT / "projects" / "correlation-diversification" / "data"
IMAGES_DIR = REPO_ROOT / "projects" / "correlation-diversification" / "images" / "heterogeneity"

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_PARQUET = DATA_DIR / "comparison_assets_monthly.parquet"
WINE_PRICE_PARQUET = DATA_DIR / "wine_price_series.parquet"

# ---------------------------------------------------------------------------
# Analysis constants
# ---------------------------------------------------------------------------
# Target wines: LWIN7 -> display label
TARGET_WINES = {
    1807626: "Salon",
    1082656: "Dom Perignon",
    1011872: "Lafite",
}

# Colour palette per spec
WINE_COLORS = {
    "Salon":        "#9437ff",
    "Dom Perignon": "#83D483",
    "Lafite":       "#FFD166",
}
SP500_COLOR  = "#4D87D0"
LIVEX_COLOR  = "#EF476F"
SPARSE_COLOR = "#F78C6B"
STRESS_SHADE = "#4A4A68"

# Months with fewer than this many transactions are flagged as sparse
SPARSE_THRESHOLD = 5

# Market stress windows (inclusive)
STRESS_PERIODS = [
    ("2007-10-01", "2009-03-31", "2008 GFC"),
    ("2020-01-01", "2020-06-30", "2020 COVID"),
    ("2022-01-01", "2022-12-31", "2022 Rate Rises"),
]

print("Data directory:   ", DATA_DIR)
print("Images directory: ", IMAGES_DIR)
print("motherduck_token present:", bool(os.getenv("motherduck_token")))

## 2. MotherDuck Connection & Schema Discovery

Schema discovery runs **before** any row-level query.  
We never assume column names — all column references below are resolved dynamically.

In [ ]:
con = duckdb.connect("md:")
print("Connected to MotherDuck")

In [ ]:
# Schema: winefi.ml.ml_unified_trades_tbvm
trades_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'ml'
      AND table_name    = 'ml_unified_trades_tbvm'
    ORDER BY ordinal_position
""").df()

print("=== winefi.ml.ml_unified_trades_tbvm ===")
print(f"Total columns: {len(trades_schema)}")
display(trades_schema)

In [ ]:
col_names = trades_schema["column_name"].tolist()


def find_col(patterns, label):
    """Return first column whose name contains any pattern (case-insensitive)."""
    for pat in patterns:
        matches = [c for c in col_names if pat in c.lower()]
        if matches:
            print(f"  {label:<14} -> '{matches[0]}' (matched '{pat}')")
            return matches[0]
    raise ValueError(
        f"Cannot auto-detect '{label}' column.\n"
        f"  Tried patterns : {patterns}\n"
        f"  Available cols : {col_names}"
    )


print("Identified columns:")
date_col  = find_col(["trade_date", "transaction_date", "date", "time"],       "date")
price_col = find_col(["price_gbp", "price_per_bottle", "price", "amount"],     "price")
lwin7_col = find_col(["lwin7", "lwin_7", "lwin"],                              "lwin7")
qty_col   = find_col(["num_bottles", "quantity", "qty", "bottles", "vol"],     "quantity")
size_col  = find_col(["bottle_size", "size_ml", "format_ml", "format", "size", "ml"], "bottle size")

## 3. Build Per-Wine Monthly Price Series

Aggregation pushed entirely into SQL:
- Filter to target LWIN7s and 750 ml bottle format only
- Volume-weighted average price (VWAP) per wine per calendar month
- Trade count and total bottle volume per month
- Months with < 5 transactions are flagged as **sparse**

In [ ]:
lwin_list = ", ".join(str(k) for k in TARGET_WINES.keys())

wine_monthly_sql = f"""
    SELECT
        CAST(\"{lwin7_col}\"  AS INTEGER)                           AS lwin7,
        DATE_TRUNC('month', CAST(\"{date_col}\" AS DATE))          AS month,
        SUM(CAST(\"{price_col}\" AS DOUBLE) * CAST(\"{qty_col}\" AS DOUBLE))
            / NULLIF(SUM(CAST(\"{qty_col}\" AS DOUBLE)), 0)        AS vwap_gbp,
        COUNT(*)                                                    AS trade_count,
        SUM(CAST(\"{qty_col}\" AS DOUBLE))                         AS total_bottles
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE CAST(\"{lwin7_col}\"  AS INTEGER) IN ({lwin_list})
      AND CAST(\"{size_col}\"   AS INTEGER) = 750
      AND CAST(\"{date_col}\"   AS DATE)   >= '2000-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
"""

print("Executing SQL ...")
wine_monthly_raw = con.execute(wine_monthly_sql).df()
wine_monthly_raw["month"]     = pd.to_datetime(wine_monthly_raw["month"])
wine_monthly_raw["wine_name"] = wine_monthly_raw["lwin7"].map(TARGET_WINES)
wine_monthly_raw["is_sparse"] = wine_monthly_raw["trade_count"] < SPARSE_THRESHOLD

print(f"Rows returned: {len(wine_monthly_raw)}")
print(f"Date range:    {wine_monthly_raw['month'].min().date()} -> {wine_monthly_raw['month'].max().date()}")
print()
print("Per-wine summary:")
display(
    wine_monthly_raw.groupby("wine_name").agg(
        months_with_data=("month", "count"),
        total_trades=("trade_count", "sum"),
        total_bottles=("total_bottles", "sum"),
        sparse_months=("is_sparse", "sum"),
        median_vwap_gbp=("vwap_gbp", "median"),
    ).round(2)
)

In [ ]:
# Pivot to wide format: rows=month, columns=wine_name
wine_price       = wine_monthly_raw.pivot(index="month", columns="wine_name", values="vwap_gbp")
wine_trades_wide = wine_monthly_raw.pivot(index="month", columns="wine_name", values="trade_count")

wine_price.index.name       = "date"
wine_trades_wide.index.name = "date"
wine_price.columns.name       = None
wine_trades_wide.columns.name = None

print(f"Price series shape:       {wine_price.shape}")
print(f"Trade count series shape: {wine_trades_wide.shape}")
print()
wine_price.head()

In [ ]:
# Build combined save DataFrame: price + trade count + sparse flag per wine
save_frames = [wine_price.copy()]
for wine_name in wine_price.columns:
    tc = wine_trades_wide[wine_name].rename(f"{wine_name}_trade_count")
    sp = (wine_trades_wide[wine_name] < SPARSE_THRESHOLD).rename(f"{wine_name}_is_sparse")
    save_frames.extend([tc, sp])

wine_price_series = pd.concat(save_frames, axis=1)
wine_price_series.to_parquet(WINE_PRICE_PARQUET)

print(f"Saved -> {WINE_PRICE_PARQUET}")
print(f"Shape: {wine_price_series.shape}")
print(f"Columns: {list(wine_price_series.columns)}")
wine_price_series.head()

## 4. Load Comparison Assets

Load S&P 500 and Liv-ex indices from the parquet produced by `01_data_setup.ipynb`.
If that file does not exist (e.g., first-run), fall back to yfinance for S&P 500.

In [ ]:
if COMPARISON_PARQUET.exists():
    comp = pd.read_parquet(COMPARISON_PARQUET)
    comp.index = pd.to_datetime(comp.index)
    print(f"Loaded {COMPARISON_PARQUET.name}: {comp.shape}")
    print(f"Columns: {list(comp.columns)}")
else:
    print("comparison_assets_monthly.parquet not found — fetching S&P 500 from yfinance")
    sp_raw = yf.download("^GSPC", start="2000-01-01", progress=False, auto_adjust=False)["Close"]
    if isinstance(sp_raw, pd.DataFrame):
        sp_raw = sp_raw.squeeze()
    comp = pd.DataFrame({"sp500": sp_raw.resample("ME").last()})

if "sp500" not in comp.columns:
    raise ValueError(f"'sp500' column missing. Available: {list(comp.columns)}")

# Identify Liv-ex 100 (or best available proxy)
livex_col = next(
    (c for c in comp.columns
     if any(p in c.lower() for p in ["100", "fine", "livex_1", "liv_ex_1"])),
    None,
)
if livex_col is None:
    # Fall back to any Liv-ex column
    livex_col = next(
        (c for c in comp.columns
         if any(p in c.lower() for p in ["livex", "liv_ex", "liv-ex"])),
        None,
    )

if livex_col:
    print(f"\nLiv-ex proxy column: '{livex_col}'")
else:
    print("\nWARNING: No Liv-ex column found. Drawdown chart will use S&P 500 only.")

## 5. Per-Wine Price Series Chart

One panel per wine — VWAP price in GBP, 750ml bottles.  
Stress periods shaded. Sparse months shown as dashed line.

In [ ]:
wine_names_present = [w for w in WINE_COLORS if w in wine_price.columns]

fig, axes = plt.subplots(
    len(wine_names_present), 1,
    figsize=(14, 5 * len(wine_names_present)),
    sharex=True,
)
if len(wine_names_present) == 1:
    axes = [axes]

fig.suptitle(
    "Fine Wine VWAP Price Series (750 ml, GBP) — Salon | Dom Perignon | Lafite",
    fontsize=13, fontweight="bold", y=1.01,
)

for ax, wine_name in zip(axes, wine_names_present):
    color = WINE_COLORS[wine_name]
    price_s  = wine_price[wine_name]
    trades_s = wine_trades_wide[wine_name]
    sparse_mask = trades_s < SPARSE_THRESHOLD

    # Shade stress periods
    for p_start, p_end, _ in STRESS_PERIODS:
        ax.axvspan(pd.Timestamp(p_start), pd.Timestamp(p_end),
                   alpha=0.07, color=STRESS_SHADE, zorder=0)

    # Dense segments: solid line; sparse segments: dashed
    dense_s  = price_s.where(~sparse_mask)
    sparse_s = price_s.where(sparse_mask)

    ax.plot(dense_s.index,  dense_s.values,  color=color, linewidth=1.8,
            label=wine_name)
    ax.plot(sparse_s.index, sparse_s.values, color=color, linewidth=1.0,
            linestyle="--", alpha=0.6,
            label=f"{wine_name} (sparse < {SPARSE_THRESHOLD} trades)")

    ax.set_ylabel("VWAP (GBP / bottle)", fontsize=9)
    ax.set_title(wine_name, fontsize=11, loc="left", color=color, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"\u00a3{x:,.0f}"))
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.3)

# Add stress period labels after all data is plotted (ylim is correct)
for ax in axes:
    ylim = ax.get_ylim()
    y_label = ylim[0] + (ylim[1] - ylim[0]) * 0.04
    for p_start, p_end, p_label in STRESS_PERIODS:
        ts = pd.Timestamp(p_start)
        te = pd.Timestamp(p_end)
        ax.text(
            ts + (te - ts) / 2, y_label, p_label,
            ha="center", va="bottom", fontsize=7,
            color=STRESS_SHADE, alpha=0.8,
        )

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()

out_path = IMAGES_DIR / "wine_price_series.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved -> {out_path}")
plt.show()

## 6. Drawdown Analysis: 2008 GFC

Compare indexed performance (base = 100 at plot start) and rolling drawdown from peak  
for each wine vs S&P 500 and Liv-ex 100 during 2006–2011.

In [ ]:
GFC_START  = pd.Timestamp("2007-10-01")
GFC_END    = pd.Timestamp("2009-03-31")
PLOT_START = pd.Timestamp("2006-01-01")
PLOT_END   = pd.Timestamp("2011-12-31")


def compute_drawdown(series):
    """Rolling drawdown (%) from expanding peak."""
    rolling_max = series.expanding().max()
    return (series - rolling_max) / rolling_max * 100


# Assemble all series for comparison
all_series = {}
for wine_name in wine_price.columns:
    s = wine_price[wine_name].dropna()
    if len(s) > 0:
        all_series[wine_name] = s

all_series["S&P 500"] = comp["sp500"].dropna()
if livex_col and livex_col in comp.columns:
    all_series[f"Liv-ex ({livex_col})"] = comp[livex_col].dropna()

COLORS_MAP = {
    **{k: v for k, v in WINE_COLORS.items()},
    "S&P 500": SP500_COLOR,
}

fig, (ax_price, ax_dd) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for name, series in all_series.items():
    s = series[(series.index >= PLOT_START) & (series.index <= PLOT_END)].dropna()
    if len(s) < 3:
        continue
    color = COLORS_MAP.get(name, LIVEX_COLOR)
    lw    = 2.0 if name in WINE_COLORS else 1.5
    ls    = "-"  if name in WINE_COLORS else "--"

    # Normalize to 100 at first point in the plot window
    s_norm = s / s.iloc[0] * 100
    ax_price.plot(s_norm.index, s_norm.values, color=color,
                  linewidth=lw, linestyle=ls, label=name)

    # Rolling drawdown from peak
    dd = compute_drawdown(s)
    ax_dd.plot(dd.index, dd.values, color=color,
               linewidth=lw, linestyle=ls, label=name)

# GFC shading on both panels
for ax in [ax_price, ax_dd]:
    ax.axvspan(GFC_START, GFC_END, alpha=0.10, color="#EF476F",
               zorder=0, label="GFC (Oct 07 - Mar 09)")
    ax.axhline(0, color=STRESS_SHADE, linewidth=0.7, linestyle=":")
    ax.legend(fontsize=8, loc="lower left")
    ax.grid(axis="y", alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax_price.set_ylabel("Indexed return (start = 100)", fontsize=9)
ax_price.set_title(
    "2008 GFC: Indexed Price Comparison — Fine Wine vs Market Benchmarks",
    fontsize=11, fontweight="bold",
)
ax_dd.set_ylabel("Drawdown from peak (%)", fontsize=9)
ax_dd.set_title("Rolling Drawdown from Peak", fontsize=10, loc="left")

plt.tight_layout()
out_path = IMAGES_DIR / "gfc_drawdown_comparison.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved -> {out_path}")
plt.show()

## 7. Trade Volume Alongside Prices

Dual-axis chart per wine: VWAP price (left axis) with monthly trade count bars (right axis).  
Orange scatter marks months with < 5 transactions (sparse) to distinguish *no trades* from *price fell*.

In [ ]:
fig, axes = plt.subplots(
    len(wine_names_present), 1,
    figsize=(14, 5 * len(wine_names_present)),
    sharex=True,
)
if len(wine_names_present) == 1:
    axes = [axes]

fig.suptitle(
    "Trade Count & VWAP Price — 750 ml bottles per month",
    fontsize=13, fontweight="bold", y=1.01,
)

for ax, wine_name in zip(axes, wine_names_present):
    color    = WINE_COLORS[wine_name]
    price_s  = wine_price[wine_name]
    trades_s = wine_trades_wide[wine_name]
    sparse_mask = trades_s < SPARSE_THRESHOLD

    ax2 = ax.twinx()

    # Trade count bars on secondary axis (drawn first, behind price)
    ax2.bar(
        trades_s.dropna().index,
        trades_s.dropna().values,
        width=25, color=color, alpha=0.20, zorder=1,
        label=f"{wine_name} trade count",
    )
    ax2.set_ylabel("Trade count (# transactions)", fontsize=8, color=STRESS_SHADE)
    ax2.tick_params(axis="y", labelcolor=STRESS_SHADE, labelsize=8)

    # VWAP price line on primary axis
    ax.plot(
        price_s.dropna().index,
        price_s.dropna().values,
        color=color, linewidth=2.0, zorder=5,
        label=f"{wine_name} VWAP",
    )

    # Mark sparse months on price line
    sparse_price = price_s[sparse_mask].dropna()
    if not sparse_price.empty:
        ax.scatter(
            sparse_price.index, sparse_price.values,
            color=SPARSE_COLOR, s=20, zorder=6,
            label=f"< {SPARSE_THRESHOLD} trades (sparse)",
        )

    ax.set_ylabel("VWAP (GBP / bottle)", fontsize=9, color=color)
    ax.tick_params(axis="y", labelcolor=color)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"\u00a3{x:,.0f}"))
    ax.set_title(wine_name, fontsize=11, loc="left", color=color, fontweight="bold")
    ax.grid(axis="y", alpha=0.2)

    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")

    # Shade stress periods
    for p_start, p_end, _ in STRESS_PERIODS:
        ax.axvspan(pd.Timestamp(p_start), pd.Timestamp(p_end),
                   alpha=0.06, color=STRESS_SHADE, zorder=0)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()

out_path = IMAGES_DIR / "wine_trade_volume.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved -> {out_path}")
plt.show()

## 8. Best & Worst Performers During Market Stress Periods

Compute total return (%) over each stress window for each wine and benchmark.  
Identify best and worst performers per period.

In [ ]:
stress_results = []
for p_start, p_end, period_name in STRESS_PERIODS:
    ts, te = pd.Timestamp(p_start), pd.Timestamp(p_end)
    row = {"Period": period_name}
    for name, series in all_series.items():
        window = series[(series.index >= ts) & (series.index <= te)].dropna()
        if len(window) < 2:
            row[name] = None
        else:
            ret = (window.iloc[-1] - window.iloc[0]) / window.iloc[0] * 100
            row[name] = round(float(ret), 2)
    stress_results.append(row)

stress_df = pd.DataFrame(stress_results).set_index("Period")

print("Return (%) during each stress period:")
display(stress_df)

print()
for period_name in stress_df.index:
    row = stress_df.loc[period_name].dropna().sort_values()
    if len(row) == 0:
        print(f"{period_name}: no data")
        continue
    best  = row.idxmax()
    worst = row.idxmin()
    print(f"{period_name}:")
    print(f"  Best performer:  {best}  ({row[best]:+.1f}%)")
    print(f"  Worst performer: {worst} ({row[worst]:+.1f}%)")
    print()

In [ ]:
wine_cols  = [c for c in stress_df.columns if c in WINE_COLORS]
other_cols = [c for c in stress_df.columns if c not in WINE_COLORS]
all_cols   = wine_cols + other_cols

if stress_df.empty or not all_cols:
    print("Insufficient data for stress period chart.")
else:
    sub       = stress_df[all_cols].copy()
    n_periods = len(sub)
    n_assets  = len(all_cols)
    x         = np.arange(n_periods)
    width     = 0.7 / n_assets

    color_list = [
        WINE_COLORS.get(c, SP500_COLOR if "S&P" in c else LIVEX_COLOR)
        for c in all_cols
    ]

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, (col, col_color) in enumerate(zip(all_cols, color_list)):
        vals   = sub[col].values.astype(float)
        offset = (i - n_assets / 2 + 0.5) * width
        bars   = ax.bar(
            x + offset, vals, width=width,
            color=col_color, label=col,
            alpha=0.85, edgecolor="white", linewidth=0.5,
        )
        # Value labels on bars
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    val + (0.5 if val >= 0 else -1.5),
                    f"{val:+.0f}%",
                    ha="center",
                    va="bottom" if val >= 0 else "top",
                    fontsize=7, color=STRESS_SHADE,
                )

    ax.axhline(0, color=STRESS_SHADE, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index, fontsize=10)
    ax.set_ylabel("Return during period (%)", fontsize=10)
    ax.set_title(
        "Best & Worst Performers During Market Stress Periods",
        fontsize=12, fontweight="bold",
    )
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:+.0f}%"))

    plt.tight_layout()
    out_path = IMAGES_DIR / "stress_period_performance.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path}")
    plt.show()

## 9. Data Quality Assertions

All assertions must pass before this notebook is considered complete.

In [ ]:
errors = []

# 1. Parquet file created and non-empty
if not WINE_PRICE_PARQUET.exists():
    errors.append(f"Missing output file: {WINE_PRICE_PARQUET}")
else:
    saved = pd.read_parquet(WINE_PRICE_PARQUET)
    if len(saved) == 0:
        errors.append("wine_price_series.parquet is empty")

# 2. Schema discovery completed (all column variables are defined)
try:
    _ = date_col, price_col, lwin7_col, qty_col, size_col
except NameError as e:
    errors.append(f"Schema discovery did not complete: {e}")

# 3. At least 2 wines have > 50 transactions between 2005 and 2020
window_2005_2020 = wine_monthly_raw[
    (wine_monthly_raw["month"] >= "2005-01-01") &
    (wine_monthly_raw["month"] <= "2020-12-31")
]
trade_counts = window_2005_2020.groupby("wine_name")["trade_count"].sum()
sufficient   = int((trade_counts > 50).sum())
if sufficient < 2:
    errors.append(
        f"Only {sufficient} of {len(trade_counts)} target wines have > 50 trades "
        f"between 2005-2020 (need >= 2).\nCounts: {trade_counts.to_dict()}"
    )

# 4. Images saved to correct directory
saved_images = sorted(IMAGES_DIR.glob("*.png"))
if len(saved_images) < 3:
    errors.append(
        f"Expected >= 3 images in {IMAGES_DIR}, found {len(saved_images)}: "
        f"{[p.name for p in saved_images]}"
    )

if errors:
    for err in errors:
        print(f"FAIL: {err}")
    raise AssertionError("Data quality assertions failed — see output above")
else:
    print("All data quality assertions PASSED.")
    print(f"  wine_price_series.parquet : {len(saved)} rows, {len(saved.columns)} columns")
    print(f"  Schema discovery          : date={date_col}, price={price_col}, lwin7={lwin7_col}")
    print(f"  Wines > 50 trades (2005-2020): {sufficient}/{len(trade_counts)}")
    print(f"    {trade_counts.to_dict()}")
    print(f"  Images saved ({len(saved_images)}): {[p.name for p in saved_images]}")

## Summary

| Output | Description | Location |
|--------|-------------|----------|
| `wine_price_series.parquet` | Monthly VWAP, trade count, sparse flag per wine | `projects/correlation-diversification/data/` |
| `wine_price_series.png` | Per-wine VWAP price series with stress period shading | `projects/correlation-diversification/images/heterogeneity/` |
| `gfc_drawdown_comparison.png` | 2008 GFC drawdown comparison vs S&P 500 & Liv-ex | `projects/correlation-diversification/images/heterogeneity/` |
| `wine_trade_volume.png` | Dual-axis: VWAP price + monthly trade count | `projects/correlation-diversification/images/heterogeneity/` |
| `stress_period_performance.png` | Best & worst performers across GFC, COVID, rate rises | `projects/correlation-diversification/images/heterogeneity/` |

**MotherDuck table queried (read-only)**
- `winefi.ml.ml_unified_trades_tbvm` — schema discovered dynamically in Section 2

**Key findings** (populated after notebook runs):
- Schema discovery: see Section 2 output for actual column names used
- Per-wine heterogeneity visible in price charts
- Drawdown comparison shows divergence during GFC vs equity benchmarks
- Sparse months flagged throughout to avoid misinterpreting data gaps as price moves

All prices are in **GBP**, 750 ml bottle format only.